# Анализ журнала сделок

Ноутбук только читает `data/trade_journal.csv` и не обращается к Bitunix. Запускайте ячейки сверху вниз.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
plt.style.use("seaborn-v0_8-darkgrid")

Matplotlib is building the font cache; this may take a moment.


## 1. Загрузка CSV

Путь определяется автоматически при запуске Jupyter из корня проекта или папки `notebooks`.

In [2]:
cwd = Path.cwd().resolve()
repo_root = cwd if (cwd / "pyproject.toml").exists() else cwd.parent
journal_path = repo_root / "data" / "trade_journal.csv"

if not journal_path.exists():
    raise FileNotFoundError(f"Журнал пока не найден: {journal_path}")

journal_path

PosixPath('/Users/Skytandem/futures_bot/data/trade_journal.csv')

In [3]:
expected_columns = [
    "event_type", "status", "symbol", "side", "order_type",
    "entry_price", "quantity", "leverage", "stop_loss",
    "take_profit", "client_id", "order_id", "position_id",
    "pnl", "fee", "funding", "net_pnl", "remaining_quantity",
    "source_event_id", "event_id", "timestamp",
]
numeric_columns = [
    "entry_price", "quantity", "leverage", "stop_loss",
    "take_profit", "pnl", "fee", "funding", "net_pnl",
    "remaining_quantity",
]

journal = pd.read_csv(journal_path, dtype=str, keep_default_na=False)
for column in expected_columns:
    if column not in journal.columns:
        journal[column] = ""
for column in numeric_columns:
    journal[column] = pd.to_numeric(journal[column], errors="coerce")
journal["timestamp"] = pd.to_datetime(
    journal["timestamp"], errors="coerce", utc=True
)
journal = journal.sort_values("timestamp", na_position="last").reset_index(drop=True)

print(f"Файл: {journal_path}")
print(f"Строк: {len(journal):,}")
print(f"Период: {journal['timestamp'].min()} — {journal['timestamp'].max()}")
display(journal.tail(10))

Файл: /Users/Skytandem/futures_bot/data/trade_journal.csv
Строк: 34
Период: 2026-07-26 21:04:26.986649+00:00 — 2026-07-27 19:25:01.260692+00:00


,event_type,status,symbol,side,order_type,entry_price,quantity,leverage,risk_percent,stop_loss,take_profit,proposal_id,execution_id,user_id,mode,client_id,order_id,position_id,simulated,error,pnl,source_event_id,event_id,timestamp,fee,funding,net_pnl,remaining_quantity
24,position,UPDATE,TAOUSDT,SHORT,,NaN,0.654,10.0,,NaN,NaN,,,,,,,2168985669117975008,,,-0.063278,position:2168985669117975008:UPDATE:1785180295467,45a02d9aaa7b4735b425eba799608c71,2026-07-27 19:25:01.251929+00:00,NaN,NaN,NaN,NaN
25,tpsl,NEW,TAOUSDT,Buy,TPSL,NaN,NaN,10.0,,NaN,189.33,,,,,,528547971787039769,2168985669117975008,,,NaN,tpsl:528547971787039769:NEW:1785180295467,d27cca33c79741a199e071ce91c0da83,2026-07-27 19:25:01.253151+00:00,NaN,NaN,NaN,NaN
26,position,UPDATE,TAOUSDT,SHORT,,NaN,0.654,10.0,,NaN,NaN,,,,,,,2168985669117975008,,,-0.063278,position:2168985669117975008:UPDATE:1785180296798,5790838e78974973bb54a8697c663b90,2026-07-27 19:25:01.254084+00:00,NaN,NaN,NaN,NaN
27,tpsl,NEW,TAOUSDT,Buy,TPSL,NaN,NaN,10.0,,NaN,187.62,,,,,,5209982050814984105,2168985669117975008,,,NaN,tpsl:5209982050814984105:NEW:1785180296798,4b473eb44f1347aa88924a91d3888fe6,2026-07-27 19:25:01.254987+00:00,NaN,NaN,NaN,NaN
28,position,UPDATE,TAOUSDT,SHORT,,NaN,0.654,10.0,,NaN,NaN,,,,,,,2168985669117975008,,,-0.063278,position:2168985669117975008:UPDATE:1785180298077,fd0b05e1c19b431796cfdd3df499e275,2026-07-27 19:25:01.255994+00:00,NaN,NaN,NaN,NaN
29,tpsl,NEW,TAOUSDT,Buy,TPSL,NaN,NaN,10.0,,NaN,184.19,,,,,,5438156242163309069,2168985669117975008,,,NaN,tpsl:5438156242163309069:NEW:1785180298077,d1533993ba2b47be85fc624cde87fcf0,2026-07-27 19:25:01.257054+00:00,NaN,NaN,NaN,NaN
30,tpsl,NEW,TAOUSDT,Buy,TPSL,NaN,NaN,10.0,,NaN,177.35,,,,,,8163181547918509150,2168985669117975008,,,NaN,tpsl:8163181547918509150:NEW:1785180299307,ea4f8d0cdb8a4f5b9df6360a29fb5405,2026-07-27 19:25:01.258048+00:00,NaN,NaN,NaN,NaN
31,position,UPDATE,TAOUSDT,SHORT,,NaN,0.654,10.0,,NaN,NaN,,,,,,,2168985669117975008,,,-0.063278,position:2168985669117975008:UPDATE:1785180299307,fb6bbdab15394b80856b200680313af3,2026-07-27 19:25:01.258977+00:00,NaN,NaN,NaN,NaN
32,position,UPDATE,TAOUSDT,SHORT,,NaN,0.654,10.0,,NaN,NaN,,,,,,,2168985669117975008,,,-0.063278,position:2168985669117975008:UPDATE:1785180300598,5631a8420f634569966039529bd6c662,2026-07-27 19:25:01.259875+00:00,NaN,NaN,NaN,NaN
33,tpsl,NEW,TAOUSDT,Buy,TPSL,NaN,NaN,10.0,,NaN,163.65,,,,,,7725837327289526802,2168985669117975008,,,NaN,tpsl:7725837327289526802:NEW:1785180300598,99f6e65f21cf4b39a4b1b5fa39eb2e88,2026-07-27 19:25:01.260692+00:00,NaN,NaN,NaN,NaN


## 2. Основные показатели

Расчёты строятся по `TRADE_SUMMARY`, поэтому закрытая позиция учитывается один раз.

In [4]:
trades = journal.loc[journal["event_type"].eq("TRADE_SUMMARY")].copy()
trades = trades.drop_duplicates("source_event_id", keep="last")

closed_count = len(trades)
wins = int(trades["net_pnl"].gt(0).sum())
losses = int(trades["net_pnl"].lt(0).sum())
breakeven = int(trades["net_pnl"].eq(0).sum())
win_rate = wins / closed_count * 100 if closed_count else 0

metrics = pd.Series({
    "Закрытых сделок": closed_count,
    "Прибыльных": wins,
    "Убыточных": losses,
    "Без результата": breakeven,
    "Win rate, %": round(win_rate, 2),
    "Realized PnL": trades["pnl"].sum(min_count=1),
    "Комиссии": trades["fee"].abs().sum(min_count=1),
    "Funding": trades["funding"].sum(min_count=1),
    "Чистый PnL": trades["net_pnl"].sum(min_count=1),
    "Средний PnL": trades["net_pnl"].mean(),
    "Лучший результат": trades["net_pnl"].max(),
    "Худший результат": trades["net_pnl"].min(),
})
display(metrics.to_frame("Значение"))

,Значение
Закрытых сделок,0.0
Прибыльных,0.0
Убыточных,0.0
Без результата,0.0
"Win rate, %",0.0
Realized PnL,NaN
Комиссии,NaN
Funding,NaN
Чистый PnL,NaN
Средний PnL,NaN


## 3. Последние завершённые сделки

In [5]:
trade_columns = [
    "timestamp", "symbol", "side", "quantity", "entry_price",
    "pnl", "fee", "funding", "net_pnl", "position_id",
]
display(trades[trade_columns].sort_values("timestamp", ascending=False).head(20))

,timestamp,symbol,side,quantity,entry_price,pnl,fee,funding,net_pnl,position_id


## 4. Накопительный чистый PnL

In [6]:
equity = trades.dropna(subset=["timestamp", "net_pnl"]).sort_values("timestamp").copy()
equity["cumulative_net_pnl"] = equity["net_pnl"].cumsum()

if equity.empty:
    print("Пока нет завершённых сделок с рассчитанным net_pnl.")
else:
    ax = equity.plot(
        x="timestamp", y="cumulative_net_pnl", figsize=(12, 4),
        title="Накопительный чистый PnL", legend=False,
    )
    ax.set_xlabel("Время")
    ax.set_ylabel("PnL")
    plt.show()

Пока нет завершённых сделок с рассчитанным net_pnl.


## 5. Результаты по торговым парам

In [7]:
if trades.empty:
    print("Пока нет завершённых сделок.")
else:
    by_symbol = trades.groupby("symbol", dropna=False).agg(
        trades=("position_id", "count"),
        wins=("net_pnl", lambda values: values.gt(0).sum()),
        net_pnl=("net_pnl", "sum"),
        average_pnl=("net_pnl", "mean"),
        fees=("fee", lambda values: values.abs().sum()),
        funding=("funding", "sum"),
    )
    by_symbol["win_rate_pct"] = by_symbol["wins"] / by_symbol["trades"] * 100
    display(by_symbol.sort_values("net_pnl", ascending=False))

Пока нет завершённых сделок.


## 6. Результаты по дням

In [ ]:
daily_source = trades.dropna(subset=["timestamp"]).copy()
daily_source["day"] = daily_source["timestamp"].dt.tz_convert("Europe/Moscow").dt.date
daily = daily_source.groupby("day").agg(
    trades=("position_id", "count"),
    net_pnl=("net_pnl", "sum"),
    fees=("fee", lambda values: values.abs().sum()),
    funding=("funding", "sum"),
)
display(daily.sort_index(ascending=False).head(30))

## 7. Исполнения ENTRY, TP и SL

In [9]:
execution_mask = journal["event_type"].eq("ENTRY") | journal["event_type"].eq("SL") | journal["event_type"].str.fullmatch(r"TP[1-5]", na=False)
executions = journal.loc[execution_mask, [
    "timestamp", "event_type", "symbol", "side", "entry_price",
    "take_profit", "stop_loss", "quantity", "remaining_quantity",
    "pnl", "fee", "funding", "order_id", "position_id",
]]
display(executions.sort_values("timestamp", ascending=False).head(50))

,timestamp,event_type,symbol,side,entry_price,take_profit,stop_loss,quantity,remaining_quantity,pnl,fee,funding,order_id,position_id


## 8. Проверка качества данных

Показывает пропущенные ID, дубли событий и незаполненные итоговые показатели.

In [10]:
quality = pd.Series({
    "Дубли source_event_id": journal.loc[journal["source_event_id"].ne(""), "source_event_id"].duplicated().sum(),
    "TP/SL без order_id": executions.loc[executions["event_type"].ne("ENTRY"), "order_id"].eq("").sum(),
    "Итоги без position_id": trades["position_id"].eq("").sum(),
    "Итоги без net_pnl": trades["net_pnl"].isna().sum(),
    "Некорректные timestamp": journal["timestamp"].isna().sum(),
})
display(quality.to_frame("Количество"))

,Количество
Дубли source_event_id,0
TP/SL без order_id,0
Итоги без position_id,0
Итоги без net_pnl,0
Некорректные timestamp,0


## 9. Экспорт завершённых сделок

Раскомментируйте строку записи, если нужен отдельный компактный CSV.

In [ ]:
export_path = repo_root / "data" / "trade_summary_export.csv"
# trades[trade_columns].to_csv(export_path, index=False)
export_path